# RAGAS Experiment

Notebook eksperimen RAGAS untuk subset data MPTP.

In [ ]:
import json
from pathlib import Path


SAMPLE_SIZE = 150


def first_existing_path(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    candidate_list = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"canonical_gold_qas_MPTP.json tidak ditemukan. Cek path berikut:\n{candidate_list}")


input_path = first_existing_path(
    Path("../../canonical_gold_qas_MPTP.json"),
    Path("../canonical_gold_qas_MPTP.json"),
    Path("canonical_gold_qas_MPTP.json"),
)

output_dir = input_path.parent / "silab" / "100exp"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"canonical_gold_qas_MPTP_{SAMPLE_SIZE}_questions.json"

with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

if not isinstance(data, list):
    raise TypeError(f"Input harus list JSON, tapi dapat: {type(data).__name__}")

if len(data) < SAMPLE_SIZE:
    raise ValueError(f"Data hanya {len(data)} item, tidak cukup untuk mengambil {SAMPLE_SIZE} item.")

sample_data = []
for item in data[:SAMPLE_SIZE]:
    question = (item.get("question") or "").strip()
    if not question:
        raise ValueError("Ada item pada 150 data pertama yang tidak punya field question.")
    sample_data.append({"question": question})

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(sample_data, f, ensure_ascii=False, indent=2)

print("Input:", input_path)
print("Output:", output_path)
print("Total input:", len(data))
print("Total question sample:", len(sample_data))
print("Contoh item pertama:")
print(json.dumps(sample_data[0], ensure_ascii=False, indent=2))


## RAG Query ke Vectorstore Ayat Indonesia + Jawaban SI-LAB

Cell ini memakai field `question` dari 150 data MPTP sebagai query, mengambil top 5 ayat dari vectorstore Bahasa Indonesia, lalu mengirim pertanyaan + context ke LLM SI-LAB. Hasilnya disimpan incremental ke JSONL.

In [5]:
import json
import os
import time
from datetime import datetime, timezone, timedelta
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone
from tqdm.auto import tqdm


def configure_ssl_certificates():
    try:
        import certifi
    except ImportError:
        print(
            "Package certifi belum terpasang. Jika muncul error SSL, jalankan: "
            "conda install -n dataquran -c conda-forge certifi ca-certificates openssl"
        )
        return

    cert_path = certifi.where()
    os.environ["SSL_CERT_FILE"] = cert_path
    os.environ["REQUESTS_CA_BUNDLE"] = cert_path


configure_ssl_certificates()


BASE_URL = "https://llms.si-lab.org/api"
API_KEY = os.getenv("SILAB_API_KEY", "sk-1194efecbf0b45d6ac175dc9b06a428a")

# Ganti model di sini jika ingin menjalankan model lain, contoh: "gemma3:1b"
MODEL_NAME = "gpt-oss:120b"

EMBEDDING_MODEL = "text-embedding-3-small"
INDEX_NAME_ID = os.getenv("INDEX_NAME_AYAT_ID", "quran-ayat-id-openai")
NAMESPACE_ID = os.getenv("PINECONE_NAMESPACE_AYAT_ID", "ayat_id")
TOP_K = 3

LOCAL_TZ = timezone(timedelta(hours=7), "Asia/Jakarta")


def model_slug(model_name: str) -> str:
    return "".join(ch if ch.isalnum() else "_" for ch in model_name.lower()).strip("_")


def timestamp_fields():
    now_utc = datetime.now(timezone.utc)
    return {
        "timestamp_utc": now_utc.isoformat(),
        "timestamp_local": now_utc.astimezone(LOCAL_TZ).isoformat(),
    }


def first_existing_path(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    candidate_list = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"File tidak ditemukan. Cek path berikut:\n{candidate_list}")


def append_jsonl(path: Path, record: dict):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl_success_indexes(path: Path) -> set[int]:
    processed = set()
    if not path.exists():
        return processed

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            if record.get("event") == "rag_answer_generated":
                processed.add(int(record["sample_index"]))

    return processed


def get_match_value(match, key, default=None):
    if isinstance(match, dict):
        return match.get(key, default)
    return getattr(match, key, default)


def build_context(sources: list[dict]) -> str:
    blocks = []
    for source in sources:
        blocks.append(
            f"[{source['rank']}] Surah {source.get('surah')} Ayat {source.get('ayat')} "
            f"({source.get('surah_transliteration') or '-'})\n"
            f"{source.get('content') or ''}"
        )
    return "\n\n".join(blocks)


def build_messages(question: str, context: str) -> list[dict]:
    system_prompt = """
This task processes Quranic scripture strictly for academic dataset construction and retrieval evaluation.

The content may contain references to warfare, punishment, divine judgment, or condemnation as part of religious scripture.
These references must be handled neutrally and descriptively.
The output must not promote, justify, or glorify violence.

The verse text is in Bahasa Indonesia.
The generated question and answer MUST be written strictly in Bahasa Indonesia.

Language Rules:
- The question MUST be entirely in Bahasa Indonesia.
- The answer MUST be entirely in Bahasa Indonesia.
- Do NOT use English words.
- Only JSON keys remain in English.

The verse belongs to multiple thematic categories.
These thematic paths provide contextual orientation only.
They must NOT introduce information beyond the verse text.

The question must simulate a natural user query.
The user does NOT see the verse.

Strict Rules:
- Do NOT refer to "ayat ini" or similar phrases.
- Do NOT assume user sees the verse.
- The answer must be supported strictly by the verse text.
- Do NOT add external information.
- If too fragmentary, return exactly: INSUFFICIENT_EVIDENCE.
- Return strictly valid JSON only.

Difficulty Classification:
- easy: short verse, single concept.
- medium: multiple concepts.
- hard: long, legal, numerical, conditional, metaphorical.

For this RAG experiment:
- The question is already provided. Do NOT generate a new question.
- Answer the provided question using only the retrieved verse context.
- Return JSON only with this schema: {"answer": "..."}.
- If the retrieved context is insufficient, return JSON only as: {"answer": "INSUFFICIENT_EVIDENCE"}.
""".strip()

    user_prompt = f"""
Pertanyaan:
{question}

Context ayat hasil retrieval:
{context}

Jawaban dalam JSON:
""".strip()

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


def call_silab_llm(messages: list[dict], max_retries: int = 5):
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            started = time.perf_counter()
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                temperature=0.0,
            )
            duration = time.perf_counter() - started
            return response, duration, attempt
        except Exception as exc:
            last_error = exc
            wait_seconds = min(60, 2 ** attempt)
            print(f"Retry LLM attempt {attempt}/{max_retries}: {exc}. Wait {wait_seconds}s")
            time.sleep(wait_seconds)

    raise RuntimeError(f"LLM gagal setelah {max_retries} retry: {last_error}")


# Load .env dari folder eksperimen, parent, atau root project.
for env_candidate in [Path.cwd() / ".env", Path.cwd().parent / ".env", Path.cwd().parent.parent / ".env"]:
    if env_candidate.exists():
        load_dotenv(env_candidate)
        print("ENV loaded from:", env_candidate.resolve())
        break
else:
    load_dotenv()
    print("File .env tidak ditemukan di cwd/parent. Mencoba load dari environment aktif.")

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY belum ada di .env atau environment.")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY belum ada di .env atau environment.")

sample_path = first_existing_path(
    Path("canonical_gold_qas_MPTP_150_questions.json"),
    Path("silab/100exp/canonical_gold_qas_MPTP_150_questions.json"),
    Path("100exp/canonical_gold_qas_MPTP_150_questions.json"),
)

lookup_path = first_existing_path(
    Path("../ayat_id_content_lookup.json"),
    Path("silab/ayat_id_content_lookup.json"),
    Path("ayat_id_content_lookup.json"),
)

output_dir = sample_path.parent
output_path = output_dir / f"for-ragas_mptp_150_id_vector_{model_slug(MODEL_NAME)}.jsonl"

with sample_path.open("r", encoding="utf-8") as f:
    samples = json.load(f)

with lookup_path.open("r", encoding="utf-8") as f:
    content_lookup = json.load(f)

processed_indexes = load_jsonl_success_indexes(output_path)

client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
)

embedding = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_key=OPENAI_API_KEY,
)

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME_ID)

print("Sample:", sample_path)
print("Output:", output_path)
print("Model:", MODEL_NAME)
print("Vector index:", INDEX_NAME_ID)
print("Namespace:", NAMESPACE_ID)
print("Top K:", TOP_K)
print("Total question samples:", len(samples))
print("Already processed:", len(processed_indexes))

for sample_index, sample in tqdm(list(enumerate(samples)), desc="RAG SI-LAB 150 MPTP"):
    if sample_index in processed_indexes:
        continue

    question = sample.get("question")
    if not question:
        append_jsonl(output_path, {
            **timestamp_fields(),
            "event": "skipped_missing_question",
            "sample_index": sample_index,
            "sample": sample,
        })
        continue

    total_started = time.perf_counter()

    retrieval_started = time.perf_counter()
    query_vector = embedding.embed_query(question)
    response = index.query(
        vector=query_vector,
        namespace=NAMESPACE_ID,
        top_k=TOP_K,
        include_values=False,
        include_metadata=True,
    )
    retrieval_duration = time.perf_counter() - retrieval_started

    matches = response.get("matches", []) if isinstance(response, dict) else getattr(response, "matches", [])
    sources = []

    for rank, match in enumerate(matches, start=1):
        doc_id = get_match_value(match, "id")
        score = get_match_value(match, "score")
        metadata = get_match_value(match, "metadata", {}) or {}
        lookup_item = content_lookup.get(doc_id, {})
        lookup_metadata = lookup_item.get("metadata", {}) or {}

        sources.append({
            "rank": rank,
            "id": doc_id,
            "score": score,
            "surah": metadata.get("surah") or lookup_metadata.get("surah"),
            "ayat": metadata.get("ayat") or lookup_metadata.get("ayat"),
            "surah_ayat": metadata.get("surah_ayat") or lookup_metadata.get("surah_ayat"),
            "surah_transliteration": metadata.get("surah_transliteration") or lookup_metadata.get("surah_transliteration"),
            "content": lookup_item.get("content", ""),
            "metadata": metadata,
        })

    context = build_context(sources)
    messages = build_messages(question, context)
    llm_response, llm_duration, llm_attempts = call_silab_llm(messages)
    answer_raw = llm_response.choices[0].message.content
    try:
        answer_json = json.loads(answer_raw)
    except json.JSONDecodeError:
        answer_json = None

    answer = answer_json.get("answer") if isinstance(answer_json, dict) else answer_raw
    usage = getattr(llm_response, "usage", None)

    record = {
        **timestamp_fields(),
        "event": "rag_answer_generated",
        "sample_index": sample_index,
        "model": MODEL_NAME,
        "question": question,
        "retrieval": {
            "embedding_model": EMBEDDING_MODEL,
            "index_name": INDEX_NAME_ID,
            "namespace": NAMESPACE_ID,
            "top_k": TOP_K,
            "duration_seconds": retrieval_duration,
        },
        "sources": sources,
        "context": context,
        "answer": answer,
        "answer_raw": answer_raw,
        "answer_json": answer_json,
        "timing": {
            "retrieval_seconds": retrieval_duration,
            "llm_seconds": llm_duration,
            "total_seconds": time.perf_counter() - total_started,
        },
        "llm_attempts": llm_attempts,
        "usage": usage.model_dump() if hasattr(usage, "model_dump") else (dict(usage) if usage else None),
    }

    append_jsonl(output_path, record)
    processed_indexes.add(sample_index)

print("Selesai.")
print("Output tersimpan di:", output_path)
print("Total processed sekarang:", len(processed_indexes))


ENV loaded from: /home/rinal/rinal/python-projects/quranrag/quranqadataset/.env
Sample: /home/rinal/rinal/python-projects/quranrag/quranqadataset/silab/100exp/canonical_gold_qas_MPTP_150_questions.json
Output: /home/rinal/rinal/python-projects/quranrag/quranqadataset/silab/100exp/for-ragas_mptp_150_id_vector_gpt_oss_120b.jsonl
Model: gpt-oss:120b
Vector index: quran-ayat-id-openai
Namespace: ayat_id
Top K: 3
Total question samples: 150
Already processed: 0


RAG SI-LAB 150 MPTP: 100%|██████████| 150/150 [17:49<00:00,  7.13s/it] 

Selesai.
Output tersimpan di: /home/rinal/rinal/python-projects/quranrag/quranqadataset/silab/100exp/for-ragas_mptp_150_id_vector_gpt_oss_120b.jsonl
Total processed sekarang: 150


## Context Relevance RAGAS

Cell ini membaca hasil RAG JSONL, mengambil `question` dan top 3 context ayat Indonesia, lalu menghitung skor `ContextRelevance`.

In [6]:
import json
import os
import time
from datetime import datetime, timezone, timedelta
from pathlib import Path

from dotenv import load_dotenv
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextRelevance
from tqdm.auto import tqdm


def configure_ssl_certificates():
    try:
        import certifi
    except ImportError:
        print(
            "Package certifi belum terpasang. Jika muncul error SSL, jalankan: "
            "conda install -n dataquran -c conda-forge certifi ca-certificates openssl"
        )
        return

    cert_path = certifi.where()
    os.environ["SSL_CERT_FILE"] = cert_path
    os.environ["REQUESTS_CA_BUNDLE"] = cert_path


configure_ssl_certificates()


RAGAS_EVAL_MODEL = os.getenv("RAGAS_EVAL_MODEL", "gpt-4o-mini")
TOP_K_CONTEXT_RELEVANCE = 3
LOCAL_TZ = timezone(timedelta(hours=7), "Asia/Jakarta")


def timestamp_fields():
    now_utc = datetime.now(timezone.utc)
    return {
        "timestamp_utc": now_utc.isoformat(),
        "timestamp_local": now_utc.astimezone(LOCAL_TZ).isoformat(),
    }


def first_existing_path(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    candidate_list = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"File tidak ditemukan. Cek path berikut:\n{candidate_list}")


def append_jsonl(path: Path, record: dict):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            records.append(json.loads(line))
    return records


def load_processed_indexes(path: Path) -> set[int]:
    processed = set()
    if not path.exists():
        return processed

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            if record.get("event") == "context_relevance_scored":
                processed.add(int(record["sample_index"]))

    return processed


def result_to_dict(result):
    if hasattr(result, "model_dump"):
        return result.model_dump()
    if hasattr(result, "dict"):
        return result.dict()
    if isinstance(result, dict):
        return result
    return {"value": getattr(result, "value", result)}


# Load .env dari folder eksperimen, parent, atau root project.
for env_candidate in [Path.cwd() / ".env", Path.cwd().parent / ".env", Path.cwd().parent.parent / ".env"]:
    if env_candidate.exists():
        load_dotenv(env_candidate)
        print("ENV loaded from:", env_candidate.resolve())
        break
else:
    load_dotenv()
    print("File .env tidak ditemukan di cwd/parent. Mencoba load dari environment aktif.")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY belum ada di .env atau environment untuk evaluator RAGAS.")

rag_output_path = first_existing_path(
    Path("for-ragas_mptp_150_id_vector_gpt_oss_120b.jsonl"),
    Path("silab/100exp/for-ragas_mptp_150_id_vector_gpt_oss_120b.jsonl"),
    Path("100exp/for-ragas_mptp_150_id_vector_gpt_oss_120b.jsonl"),
)

score_output_path = rag_output_path.with_name(
    f"context_relevance_{rag_output_path.stem}_k{TOP_K_CONTEXT_RELEVANCE}_{RAGAS_EVAL_MODEL.replace('-', '_')}.jsonl"
)

rag_records = [
    record
    for record in load_jsonl(rag_output_path)
    if record.get("event") == "rag_answer_generated"
]
processed_indexes = load_processed_indexes(score_output_path)

client = AsyncOpenAI()
llm = llm_factory(RAGAS_EVAL_MODEL, client=client)
scorer = ContextRelevance(llm=llm)

print("Input RAG:", rag_output_path)
print("Output score:", score_output_path)
print("Evaluator model:", RAGAS_EVAL_MODEL)
print("K context:", TOP_K_CONTEXT_RELEVANCE)
print("Total RAG records:", len(rag_records))
print("Already scored:", len(processed_indexes))

for record in tqdm(rag_records, desc="ContextRelevance"):
    sample_index = int(record["sample_index"])
    if sample_index in processed_indexes:
        continue

    question = record["question"]
    top_sources = (record.get("sources") or [])[:TOP_K_CONTEXT_RELEVANCE]
    retrieved_contexts = [
        (source.get("content") or "").strip()
        for source in top_sources
        if (source.get("content") or "").strip()
    ]

    if not retrieved_contexts:
        append_jsonl(score_output_path, {
            **timestamp_fields(),
            "event": "context_relevance_error",
            "sample_index": sample_index,
            "question": question,
            "error": "Tidak ada retrieved_contexts yang valid.",
        })
        continue

    started = time.perf_counter()
    result = await scorer.ascore(
        user_input=question,
        retrieved_contexts=retrieved_contexts,
    )
    duration = time.perf_counter() - started
    result_dict = result_to_dict(result)

    scored_record = {
        **timestamp_fields(),
        "event": "context_relevance_scored",
        "sample_index": sample_index,
        "question": question,
        "retrieved_contexts": retrieved_contexts,
        "sources": top_sources,
        "context_relevance_score": result_dict.get("value", getattr(result, "value", None)),
        "context_relevance_result": result_dict,
        "rag_answer": record.get("answer"),
        "rag_answer_raw": record.get("answer_raw"),
        "evaluator_model": RAGAS_EVAL_MODEL,
        "k": TOP_K_CONTEXT_RELEVANCE,
        "duration_seconds": duration,
    }

    append_jsonl(score_output_path, scored_record)
    processed_indexes.add(sample_index)

print("Selesai scoring ContextRelevance.")
print("Output tersimpan di:", score_output_path)
print("Total scored sekarang:", len(processed_indexes))


ENV loaded from: /home/rinal/rinal/python-projects/quranrag/quranqadataset/.env
Input RAG: /home/rinal/rinal/python-projects/quranrag/quranqadataset/silab/100exp/for-ragas_mptp_150_id_vector_gpt_oss_120b.jsonl
Output score: /home/rinal/rinal/python-projects/quranrag/quranqadataset/silab/100exp/context_relevance_for-ragas_mptp_150_id_vector_gpt_oss_120b_k3_gpt_4o_mini.jsonl
Evaluator model: gpt-4o-mini
K context: 3
Total RAG records: 150
Already scored: 0


ContextRelevance: 100%|██████████| 150/150 [1:47:53<00:00, 43.16s/it]

Selesai scoring ContextRelevance.
Output tersimpan di: /home/rinal/rinal/python-projects/quranrag/quranqadataset/silab/100exp/context_relevance_for-ragas_mptp_150_id_vector_gpt_oss_120b_k3_gpt_4o_mini.jsonl
Total scored sekarang: 150
